# Deep Past Challenge - ByT5 + Ensemble MBR Inference v2

**Target: Score >45** (GeoMean of BLEU × chrF++ ≥ 2025)

### Key Improvements over v1
1. **Massive MBR pool**: 8 beam + 16 diverse sample candidates = 24 per model (was 6)
2. **Geo-mean utility**: MBR selects on `sqrt(BLEU × chrF++)` matching competition metric
3. **Multi-model ensemble**: Pool candidates from ALL attached models, MBR across combined pool
4. **Diverse sampling**: Multiple temperatures (0.5, 0.8, 1.0) and top_p (0.85, 0.95) for diversity
5. **Fixed postprocessing**: Unicode fraction chars, self-tested

### Models to Attach (attach as many as available)
1. `mattiaangeli/byt5-akkadian-mbr-v2` — ByT5-large 582M (baseline 35.5)
2. `manwithacat/akkadian-t5-enriched-v2` — T5 variant (different errors = better ensemble)
3. Our fine-tuned model — ByT5-large fine-tuned on OA data

### Requirements
GPU T4x2 | Internet OFF | Runtime < 9 hours

In [1]:
#!pip install -q sacrebleu

import os, re, json, time, warnings, logging, random
from pathlib import Path
from dataclasses import dataclass
from typing import List
from contextlib import nullcontext

import pandas as pd
import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader, Sampler
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
from tqdm.auto import tqdm
import sacrebleu

warnings.filterwarnings('ignore')

print(f"PyTorch: {torch.__version__}")
print(f"CUDA: {torch.cuda.is_available()} | GPUs: {torch.cuda.device_count()}")
for i in range(torch.cuda.device_count()):
    n = torch.cuda.get_device_name(i)
    m = torch.cuda.get_device_properties(i).total_memory / 1e9
    print(f"  GPU {i}: {n} ({m:.1f}GB)")

PyTorch: 2.9.0+cu126
CUDA: True | GPUs: 2
  GPU 0: Tesla T4 (15.6GB)
  GPU 1: Tesla T4 (15.6GB)


In [2]:
# ============================================================
# CONFIGURATION
# ============================================================
import math

# === Discover ALL available models ===
# Attach multiple models for ensemble MBR
MODEL_SEARCH_PATHS = [
    # Our fine-tuned model (top priority)
    "/kaggle/input/models/manish756/akkadian-models/pytorch/default/2/model/final",
    # mattiaangeli's model
    "/kaggle/input/models/mattiaangeli/byt5-akkadian-mbr/pytorch/default/6",
    # manwithacat's models
    "/kaggle/input/models/manwithacat/akkadian-byt5-synth-v1-0-0/pytorch/default/1",
    
    # Old mt5 model (include for ensemble diversity)
    "/kaggle/input/datasets/manish756/akkadian-additional-data-model/kaggle/working/akkadian_model/checkpoint-37680",
    # Local testing
    "../models/byt5-akkadian/final",
]

DISCOVERED_MODELS = []
seen_paths = set()
for p in MODEL_SEARCH_PATHS:
    if Path(p).exists() and p not in seen_paths:
        # Verify it has model files
        files = list(Path(p).glob('*.safetensors')) + list(Path(p).glob('*.bin'))
        config = Path(p) / 'config.json'
        if files or config.exists():
            DISCOVERED_MODELS.append(p)
            seen_paths.add(p)

assert len(DISCOVERED_MODELS) > 0, "No models found! Attach at least one model dataset."

TEST_PATH = "/kaggle/input/competitions/deep-past-initiative-machine-translation/test.csv"
if not Path(TEST_PATH).exists():
    TEST_PATH = "../data/deep-past-initiative-machine-translation/test.csv"

OUTPUT_DIR = "/kaggle/working/" if Path("/kaggle").exists() else "./output/"
Path(OUTPUT_DIR).mkdir(parents=True, exist_ok=True)

# === Generation params ===
MAX_LENGTH = 384
NUM_BEAMS = 8
MAX_NEW_TOKENS = 256
LENGTH_PENALTY = 1.3
REPETITION_PENALTY = 1.2

# === Batch sizes - auto-detect based on GPU ===
if torch.cuda.is_available():
    vram = torch.cuda.get_device_properties(0).total_memory / 1e9
    gpu_name = torch.cuda.get_device_name(0)
    if vram >= 30:  # A100/H100
        BATCH_SIZE = 8
    elif vram >= 14:  # T4 (16GB)
        BATCH_SIZE = 2  # Conservative for 8-beam + samples
    else:
        BATCH_SIZE = 1
else:
    BATCH_SIZE = 1

NUM_WORKERS = 2

# === MBR config ===
# Scale candidates based on available compute
# T4x2 has ~9h runtime limit; need to balance quality vs time
USE_MBR = True
MBR_NUM_BEAM_CANDS = 8

# Diverse sampling configs: (temperature, top_p, count)
# 4 configs × 4 samples = 16 diverse samples + 8 beam = 24 total per model
MBR_SAMPLE_CONFIGS = [
    (0.5, 0.85, 4),   # Low temp, focused
    (0.7, 0.90, 4),   # Medium temp
    (1.0, 0.95, 4),   # High temp, diverse
    (0.8, 0.80, 4),   # Medium temp, more focused
]

PREFIX = "translate Akkadian to English: "

total_per_model = MBR_NUM_BEAM_CANDS + sum(c[2] for c in MBR_SAMPLE_CONFIGS)
total_all = total_per_model * len(DISCOVERED_MODELS)

print(f"Discovered {len(DISCOVERED_MODELS)} model(s):")
for i, m in enumerate(DISCOVERED_MODELS):
    print(f"  [{i}] {m}")
print(f"\nMBR: {MBR_NUM_BEAM_CANDS} beam + {sum(c[2] for c in MBR_SAMPLE_CONFIGS)} diverse = {total_per_model} cands/model × {len(DISCOVERED_MODELS)} models = {total_all} total")
print(f"Batch size: {BATCH_SIZE}, Beams: {NUM_BEAMS}, length_penalty: {LENGTH_PENALTY}")

Discovered 3 model(s):
  [0] /kaggle/input/models/manish756/akkadian-models/pytorch/default/2/model/final
  [1] /kaggle/input/models/mattiaangeli/byt5-akkadian-mbr/pytorch/default/6
  [2] /kaggle/input/datasets/manish756/akkadian-additional-data-model/kaggle/working/akkadian_model/checkpoint-37680

MBR: 8 beam + 16 diverse = 24 cands/model × 3 models = 72 total
Batch size: 2, Beams: 8, length_penalty: 1.3


In [3]:
# ============================================================
# PREPROCESSING - Minimal (matched to model training)
# ============================================================
# Critical insight: the high-scoring model uses MINIMAL preprocessing.
# Only gap normalization. No determinative conversion, no H replacement.
# The model learns to handle raw ATF notation internally.

_RE_BIG_GAP = re.compile(r'(\.{3,}|\u2026+|\u2026\u2026)')
_RE_SMALL_GAP = re.compile(r'(xx+|\s+x\s+)')

def preprocess(text):
    """V3-aligned preprocessing: unify gaps to <gap> only, ḫ→h."""
    if pd.isna(text):
        return ''
    text = str(text)
    # ḫ/Ḫ → h/H (test set uses h/H only)
    text = text.replace('ḫ', 'h').replace('Ḫ', 'H')
    # Gap normalization: all → <gap> (v3 unified)
    text = _RE_BIG_GAP.sub('<gap>', text)
    text = _RE_SMALL_GAP.sub('<gap>', text)
    text = text.replace('<big_gap>', '<gap>')
    # Deduplicate sequential gaps
    text = re.sub(r'(<gap>[\s\-]*){2,}', '<gap> ', text)
    return text

def preprocess_batch(texts):
    s = pd.Series(texts).fillna('').astype(str)
    # V3: all gaps → <gap> (no <big_gap>), plus ḫ→h
    s = s.str.replace('ḫ', 'h', regex=False).str.replace('Ḫ', 'H', regex=False)
    s = s.str.replace(_RE_BIG_GAP, '<gap>', regex=True)
    s = s.str.replace(_RE_SMALL_GAP, '<gap>', regex=True)
    s = s.str.replace('<big_gap>', '<gap>', regex=False)
    return s.tolist()

print("Preprocessing defined.")

Preprocessing defined.


In [4]:
# ============================================================
# POSTPROCESSING - Aggressive (matched to 35.5 approach)
# ============================================================
# BUG FIX: \u00bd in raw strings (r'...') becomes literal \u which
# crashes re.sub. Use actual Unicode chars ½ ¼ ¾ directly.

_PP_PATTERNS = {
    'gap': re.compile(r'(\[x\]|\(x\)|\bx\b)', re.I),
    'big_gap': re.compile(r'(\.{3,}|\u2026|\[\.+\])'),
    'annotations': re.compile(r'\((fem|plur|pl|sing|singular|plural|\?|!)\.*\s*\w*\)', re.I),
    'repeated_words': re.compile(r'\b(\w+)(?:\s+\1\b)+'),
    'whitespace': re.compile(r'\s+'),
    'punct_space': re.compile(r'\s+([.,:])'),
    'repeated_punct': re.compile(r'([.,])\1+'),
}

_SUBSCRIPT_TRANS = str.maketrans(
    '\u2080\u2081\u2082\u2083\u2084\u2085\u2086\u2087\u2088\u2089',
    '0123456789'
)
_SPECIAL_TRANS = str.maketrans('\u1e2b\u1e2a', 'hH')

# Forbidden chars to remove (gaps are protected first)
_FORBIDDEN = '!?()"\u2014\u2014<>\u2308\u2309\u230a\u230b[]+\u02be/;'
_FORBIDDEN_TRANS = str.maketrans('', '', _FORBIDDEN)

_NGRAM_PATS = []
for n in range(4, 1, -1):
    p = r'\b((?:\w+\s+){' + str(n-1) + r'}\w+)(?:\s+\1\b)+'
    _NGRAM_PATS.append(re.compile(p))


def postprocess_batch(translations):
    """Postprocess translations with vectorized pandas operations."""
    s = pd.Series(translations).fillna('').astype(str)
    
    # Basic cleaning
    s = s.str.translate(_SPECIAL_TRANS)
    s = s.str.translate(_SUBSCRIPT_TRANS)
    s = s.str.replace(_PP_PATTERNS['whitespace'], ' ', regex=True).str.strip()
    
    # Gap normalization
    s = s.str.replace(_PP_PATTERNS['gap'], '<gap>', regex=True)
    s = s.str.replace(_PP_PATTERNS['big_gap'], '<big_gap>', regex=True)
    # v3: unify all gaps to <gap> only (no <big_gap>)
    s = s.str.replace('<big_gap>', '<gap>', regex=False)
    s = s.str.replace('<gap> <gap>', '<gap>', regex=False)
    
    # Remove annotations like (fem), (plur), etc.
    s = s.str.replace(_PP_PATTERNS['annotations'], '', regex=True)
    


    
    # Fractions - MUST come BEFORE forbidden char removal (/ is in _FORBIDDEN!)
    # Order: specific 0.X patterns FIRST, then generic (\d+).X patterns
    s = s.str.replace(r'\b0\.5\b',    '½',        regex=True)
    s = s.str.replace(r'(\d+)\.5\b',  '\\g<1>½',  regex=True)
    s = s.str.replace(r'\b0\.25\b',   '¼',        regex=True)
    s = s.str.replace(r'(\d+)\.25\b', '\\g<1>¼',  regex=True)
    s = s.str.replace(r'\b0\.75\b',   '¾',        regex=True)
    s = s.str.replace(r'(\d+)\.75\b', '\\g<1>¾',  regex=True)
    
    # Also handle 1/3 and 2/3
    s = s.str.replace(r'\b1/3\b', '⅓', regex=True)
    s = s.str.replace(r'\b2/3\b', '⅔', regex=True)
    s = s.str.replace(r'\b1/6\b', '⅙', regex=True)
    s = s.str.replace(r'\b5/6\b', '⅚', regex=True)
    
    
    # NOW remove forbidden chars (fractions already converted above)
    s = s.str.replace('<gap>', '\x00GAP\x00', regex=False)
        s = s.str.translate(_FORBIDDEN_TRANS)
    s = s.str.replace('\x00GAP\x00', ' <gap> ', regex=False)
        
    # Repeated words and n-grams
    s = s.str.replace(_PP_PATTERNS['repeated_words'], r'\1', regex=True)
    for pat in _NGRAM_PATS:
        s = s.str.replace(pat, r'\1', regex=True)
    
    # Fix punctuation
    s = s.str.replace(_PP_PATTERNS['punct_space'], r'\1', regex=True)
    s = s.str.replace(_PP_PATTERNS['repeated_punct'], r'\1', regex=True)
    s = s.str.replace(_PP_PATTERNS['whitespace'], ' ', regex=True)
    s = s.str.strip().str.strip('-').str.strip()
    
    return s.tolist()


# Quick self-test to verify no regex errors
_test = postprocess_batch([
    "Seal of Mannum-balum-Aššur son of Ṣill-Adad.",
    "He paid 1.5 minas of silver.",
    "0.5 shekel and 1/3 mina.",
])
assert len(_test) == 3 and all(len(t) > 0 for t in _test), f"Postprocess self-test FAILED: {_test}"
print(f"Postprocessing defined and self-tested OK:")
for t in _test:
    print(f"  → {t}")

Postprocessing defined and self-tested OK:
  → Seal of Mannum-balum-Aššur son of Ṣill-Adad.
  → He paid 1½ minas of silver.
  → ½ shekel and 13 mina.


In [5]:
# ============================================================
# MBR DECODING - with Geo-Mean Utility
# ============================================================
# KEY INSIGHT: Competition metric is sqrt(BLEU * chrF++).
# Using chrF++ alone as MBR utility biases toward chrF++ at the expense of BLEU.
# Using geo-mean utility directly optimizes what gets measured.

_CHRFPP = sacrebleu.metrics.CHRF(word_order=2)
_BLEU = sacrebleu.metrics.BLEU(effective_order=True)  # smoothed sentence BLEU

def _sim_chrf(a, b):
    """chrF++ similarity between two strings."""
    a = (a or '').strip()
    b = (b or '').strip()
    if not a or not b:
        return 0.0
    return float(_CHRFPP.sentence_score(a, [b]).score)

def _sim_geomean(a, b):
    """Geometric mean of BLEU and chrF++ - matches competition metric."""
    a = (a or '').strip()
    b = (b or '').strip()
    if not a or not b:
        return 0.0
    bleu = float(_BLEU.sentence_score(a, [b]).score)
    chrf = float(_CHRFPP.sentence_score(a, [b]).score)
    # Use max(x, 0.1) to avoid zero-product killing the geo-mean
    return math.sqrt(max(bleu, 0.1) * max(chrf, 0.1))


def mbr_pick(cands, utility='geomean'):
    """Select best candidate using MBR with specified utility function.
    
    With 24+ candidates, this finds the consensus translation that
    maximizes average similarity to all other candidates.
    """
    # Deduplicate while preserving order
    seen = set()
    unique = []
    for c in cands:
        c = str(c).strip()
        if c and c not in seen:
            unique.append(c)
            seen.add(c)
    
    if len(unique) == 0: return ''
    if len(unique) == 1: return unique[0]
    
    # Select similarity function
    sim_fn = _sim_geomean if utility == 'geomean' else _sim_chrf
    
    # Precompute all pairwise similarities (O(n^2) but n is small)
    n = len(unique)
    scores = [0.0] * n
    for i in range(n):
        for j in range(n):
            if i != j:
                scores[i] += sim_fn(unique[i], unique[j])
        scores[i] /= max(1, n - 1)
    
    best_i = max(range(n), key=lambda i: scores[i])
    return unique[best_i]


# Quick self-test
_test_cands = [
    "Seal of Mannum-balum-Aššur, son of Ṣilli-Adad.",
    "Seal of Mannum-balum-Aššur son of Ṣill-Adad.",
    "The seal of Mannum-balum-Aššur, son of Ṣilli-Adad.",
    "Mannum-balum-Aššur son of Ṣill-Adad seal.",
]
_pick = mbr_pick(_test_cands)
print(f"MBR defined (geo-mean utility). Self-test: '{_pick}'")

MBR defined (geo-mean utility). Self-test: 'Seal of Mannum-balum-Aššur, son of Ṣilli-Adad.'


In [6]:
# ============================================================
# LOAD MODEL(S) - Multi-model ensemble support
# ============================================================
import gc

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# We load models one at a time during inference to conserve VRAM.
# Store configs here, load weights on-demand.
model_configs = []
for model_path in DISCOVERED_MODELS:
    try:
        # Quick-load just config to check model type
        from transformers import AutoConfig
        cfg = AutoConfig.from_pretrained(model_path)
        model_type = cfg.model_type  # 't5', 'mt5', etc.
        n_params = sum(
            cfg.d_model * cfg.d_ff * cfg.num_layers * 4  # rough estimate
            for _ in [1]
        ) if hasattr(cfg, 'd_model') else 0
        
        # Determine prefix based on model type
        prefix = PREFIX  # default: "translate Akkadian to English: "
        
        model_configs.append({
            'path': model_path,
            'type': model_type,
            'prefix': prefix,
            'name': Path(model_path).parent.name + '/' + Path(model_path).name,
        })
        print(f"  ✓ {model_configs[-1]['name']} ({model_type})")
    except Exception as e:
        print(f"  ✗ {model_path}: {e}")

# Load first model (primary) - keep in memory
print(f"\nLoading primary model: {model_configs[0]['path']}")
primary_tokenizer = AutoTokenizer.from_pretrained(model_configs[0]['path'])
primary_model = AutoModelForSeq2SeqLM.from_pretrained(model_configs[0]['path'])
primary_model = primary_model.to(device).eval()

# Try BetterTransformer for speed
try:
    from optimum.bettertransformer import BetterTransformer
    primary_model = BetterTransformer.transform(primary_model)
    print('  BetterTransformer applied')
except:
    pass

if device.type == 'cuda':
    try:
        torch.set_float32_matmul_precision('high')
    except:
        pass

n_params = sum(p.numel() for p in primary_model.parameters())
print(f"  Loaded: {n_params/1e6:.0f}M params on {device}")
print(f"\nTotal models for ensemble: {len(model_configs)}")

  ✓ model/final (t5)
  ✓ default/6 (t5)
  ✓ akkadian_model/checkpoint-37680 (mt5)

Loading primary model: /kaggle/input/models/manish756/akkadian-models/pytorch/default/2/model/final


Loading weights:   0%|          | 0/500 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie shared.weight to encoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie shared.weight to decoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


  Loaded: 1229M params on cuda

Total models for ensemble: 3


In [7]:
# ============================================================
# LOAD & PREPROCESS TEST DATA
# ============================================================

# Use primary tokenizer for data loading
tokenizer = primary_tokenizer

test_df = pd.read_csv(TEST_PATH)
print(f"Test: {len(test_df)} samples")

# Minimal preprocessing
raw_texts = test_df['transliteration'].tolist()
processed = preprocess_batch(raw_texts)
test_df['input_text'] = [PREFIX + t for t in processed]
test_df['tlen'] = test_df['input_text'].str.split().str.len()

print(f"\nLength distribution:")
print(f"  Mean: {test_df['tlen'].mean():.0f} words")
print(f"  Median: {test_df['tlen'].median():.0f} words")
print(f"  Max: {test_df['tlen'].max()} words")

print(f"\nFirst 3 samples:")
for _, row in test_df.head(3).iterrows():
    print(f"  [{row['id']}] {row['input_text'][:80]}")

Test: 4 samples

Length distribution:
  Mean: 25 words
  Median: 22 words
  Max: 38 words

First 3 samples:
  [0] translate Akkadian to English: um-ma kà-ru-um kà-ni-ia-ma a-na aa-qí-il<big_gap>
  [1] translate Akkadian to English: i-na mup-pì-im aa a-lim(ki) ia-tù u„-mì-im a-nim 
  [2] translate Akkadian to English: ki-ma mup-pì-ni ta-áa-me-a-ni a-ma-kam lu a-na aí


In [8]:
# ============================================================
# BUCKET BATCH SAMPLER (groups similar lengths)
# ============================================================

class AkkDataset(Dataset):
    def __init__(self, ids, texts):
        self.ids = ids
        self.texts = texts
    def __len__(self): return len(self.ids)
    def __getitem__(self, i): return self.ids[i], self.texts[i]

class BucketSampler(Sampler):
    def __init__(self, dataset, batch_size, num_buckets=4):
        lengths = [len(t.split()) for _, t in dataset]
        sorted_idx = sorted(range(len(lengths)), key=lambda i: lengths[i])
        bsz = max(1, len(sorted_idx) // num_buckets)
        self.batches = []
        for b in range(num_buckets):
            start = b * bsz
            end = None if b == num_buckets - 1 else (b+1) * bsz
            bucket = sorted_idx[start:end]
            for i in range(0, len(bucket), batch_size):
                self.batches.append(bucket[i:i+batch_size])
    def __iter__(self):
        for b in self.batches: yield b
    def __len__(self): return len(self.batches)

def collate_fn(batch):
    ids = [s[0] for s in batch]
    texts = [s[1] for s in batch]
    tok = tokenizer(texts, max_length=MAX_LENGTH, padding=True, truncation=True, return_tensors='pt')
    return ids, tok

# Create dataset and dataloader
ds = AkkDataset(test_df['id'].tolist(), test_df['input_text'].tolist())
sampler = BucketSampler(ds, BATCH_SIZE, num_buckets=4)
loader = DataLoader(
    ds, batch_sampler=sampler, num_workers=NUM_WORKERS,
    collate_fn=collate_fn, pin_memory=True,
    prefetch_factor=2, persistent_workers=NUM_WORKERS > 0,
)
print(f"DataLoader: {len(loader)} batches")

DataLoader: 4 batches


In [9]:
# ============================================================
# INFERENCE - Multi-model Ensemble + Massive MBR
# ============================================================

def generate_candidates_for_batch(model, tokenizer, input_ids, attn_mask, device):
    """Generate a diverse candidate pool from one model for one batch."""
    B = input_ids.shape[0]
    all_pools = [[] for _ in range(B)]
    
    # 1. Beam search candidates (high quality, low diversity)
    nb = max(NUM_BEAMS, MBR_NUM_BEAM_CANDS)
    beam_out = model.generate(
        input_ids=input_ids, attention_mask=attn_mask,
        do_sample=False, num_beams=nb,
        num_return_sequences=MBR_NUM_BEAM_CANDS,
        max_new_tokens=MAX_NEW_TOKENS,
        length_penalty=LENGTH_PENALTY,
        repetition_penalty=REPETITION_PENALTY,
        early_stopping=True, use_cache=True,
    )
    beam_txt = tokenizer.batch_decode(beam_out, skip_special_tokens=True)
    for i in range(B):
        all_pools[i].extend(beam_txt[i*MBR_NUM_BEAM_CANDS:(i+1)*MBR_NUM_BEAM_CANDS])
    
    # 2. Diverse sampling candidates (varied temperatures and top_p)
    for temp, top_p, count in MBR_SAMPLE_CONFIGS:
        if count <= 0:
            continue
        samp_out = model.generate(
            input_ids=input_ids, attention_mask=attn_mask,
            do_sample=True, num_beams=1,
            temperature=temp, top_p=top_p,
            num_return_sequences=count,
            max_new_tokens=MAX_NEW_TOKENS,
            repetition_penalty=REPETITION_PENALTY,
            use_cache=True,
        )
        samp_txt = tokenizer.batch_decode(samp_out, skip_special_tokens=True)
        for i in range(B):
            all_pools[i].extend(samp_txt[i*count:(i+1)*count])
    
    return all_pools


t0 = time.time()
results = []

# --- Phase 1: Generate candidates from PRIMARY model ---
print(f"\n{'='*60}")
print(f"Phase 1: Generating candidates from primary model")
print(f"{'='*60}")

# Build candidate pools keyed by sample id
candidate_pools = {}  # id -> list of candidate strings

model = primary_model

with torch.inference_mode():
    for batch_idx, (batch_ids, tokenized) in enumerate(tqdm(loader, desc='Model 1')):
        try:
            input_ids = tokenized.input_ids.to(device)
            attn_mask = tokenized.attention_mask.to(device)
            
            if USE_MBR:
                pools = generate_candidates_for_batch(model, tokenizer, input_ids, attn_mask, device)
                for bid, pool in zip(batch_ids, pools):
                    candidate_pools.setdefault(bid, []).extend(pool)
            else:
                out = model.generate(
                    input_ids=input_ids, attention_mask=attn_mask,
                    num_beams=NUM_BEAMS, length_penalty=LENGTH_PENALTY,
                    repetition_penalty=REPETITION_PENALTY,
                    max_new_tokens=MAX_NEW_TOKENS, early_stopping=True, use_cache=True,
                )
                texts = tokenizer.batch_decode(out, skip_special_tokens=True)
                for bid, txt in zip(batch_ids, texts):
                    candidate_pools.setdefault(bid, []).append(txt)
            
            if torch.cuda.is_available() and batch_idx % 10 == 0:
                torch.cuda.empty_cache()
                
        except Exception as e:
            print(f"Batch {batch_idx} error: {e}")
            for bid in batch_ids:
                candidate_pools.setdefault(bid, []).append('')

elapsed1 = time.time() - t0
print(f"Primary model done: {elapsed1:.0f}s")
print(f"Avg candidates/sample: {np.mean([len(v) for v in candidate_pools.values()]):.1f}")

# --- Phase 2: Generate candidates from ADDITIONAL models (if any) ---
if len(model_configs) > 1:
    # Free primary model to make VRAM space
    del model
    gc.collect()
    torch.cuda.empty_cache()
    
    for model_idx in range(1, len(model_configs)):
        mcfg = model_configs[model_idx]
        print(f"\n{'='*60}")
        print(f"Phase 2.{model_idx}: Generating from {mcfg['name']}")
        print(f"{'='*60}")
        
        try:
            # Load this model
            m_tokenizer = AutoTokenizer.from_pretrained(mcfg['path'])
            m_model = AutoModelForSeq2SeqLM.from_pretrained(mcfg['path'])
            m_model = m_model.to(device).eval()
            
            try:
                from optimum.bettertransformer import BetterTransformer
                m_model = BetterTransformer.transform(m_model)
            except:
                pass
            
            # Re-tokenize with this model's tokenizer and generate
            # Process all test data through this model
            for batch_start in tqdm(range(0, len(test_df), BATCH_SIZE), desc=f'Model {model_idx+1}'):
                batch_slice = test_df.iloc[batch_start:batch_start+BATCH_SIZE]
                batch_ids = batch_slice['id'].tolist()
                batch_texts = batch_slice['input_text'].tolist()
                
                # Use this model's prefix if different
                if mcfg['prefix'] != PREFIX:
                    batch_texts = [mcfg['prefix'] + t.replace(PREFIX, '', 1) for t in batch_texts]
                
                tok = m_tokenizer(batch_texts, max_length=MAX_LENGTH, padding=True, truncation=True, return_tensors='pt')
                input_ids = tok.input_ids.to(device)
                attn_mask = tok.attention_mask.to(device)
                
                try:
                    with torch.inference_mode():
                        pools = generate_candidates_for_batch(m_model, m_tokenizer, input_ids, attn_mask, device)
                    for bid, pool in zip(batch_ids, pools):
                        candidate_pools.setdefault(bid, []).extend(pool)
                except Exception as e:
                    print(f"  Batch error: {e}")
            
            # Free this model
            del m_model, m_tokenizer
            gc.collect()
            torch.cuda.empty_cache()
            
        except Exception as e:
            print(f"  Failed to load model {mcfg['name']}: {e}")
    
    # Reload primary model for any final use
    primary_model = AutoModelForSeq2SeqLM.from_pretrained(model_configs[0]['path'])
    primary_model = primary_model.to(device).eval()

# --- Phase 3: MBR selection + postprocessing ---
print(f"\n{'='*60}")
print(f"Phase 3: MBR selection ({len(candidate_pools)} samples)")
print(f"{'='*60}")

for sid in tqdm(sorted(candidate_pools.keys()), desc='MBR'):
    cands = candidate_pools[sid]
    if USE_MBR and len(cands) > 1:
        best = mbr_pick(cands, utility='geomean')
    elif len(cands) > 0:
        best = cands[0]
    else:
        best = ''
    results.append((sid, best))

# Postprocess all at once
raw_translations = [r[1] for r in results]
cleaned = postprocess_batch(raw_translations)
results = [(results[i][0], cleaned[i]) for i in range(len(results))]

elapsed = time.time() - t0
print(f"\nTotal: {len(results)} translations in {elapsed:.0f}s ({elapsed/max(len(results),1):.1f}s/each)")
avg_cands = np.mean([len(v) for v in candidate_pools.values()])
print(f"Avg candidates per sample: {avg_cands:.1f}")


Phase 1: Generating candidates from primary model


Model 1:   0%|          | 0/4 [00:00<?, ?it/s]

Primary model done: 158s
Avg candidates/sample: 24.0

Phase 2.1: Generating from default/6


Loading weights:   0%|          | 0/252 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


...

Loading weights:   0%|          | 0/500 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie shared.weight to encoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie shared.weight to decoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning



Phase 3: MBR selection (4 samples)


MBR:   0%|          | 0/4 [00:00<?, ?it/s]


Total: 4 translations in 249s (62.3s/each)
Avg candidates per sample: 72.0


In [10]:
# ============================================================
# SUBMISSION
# ============================================================

sub = pd.DataFrame(results, columns=['id', 'translation'])
sub = sub.sort_values('id').reset_index(drop=True)
sub['translation'] = sub['translation'].fillna('')

# Validation
empty = sub['translation'].str.strip().eq('').sum()
lengths = sub['translation'].str.len()
print(f"Total: {len(sub)} translations")
print(f"Empty: {empty} ({empty/max(1,len(sub))*100:.1f}%)")
print(f"Lengths: mean={lengths.mean():.0f}, median={lengths.median():.0f}, min={lengths.min()}, max={lengths.max()}")

# Show samples
print(f"\nSamples:")
indices = [0, len(sub)//2, len(sub)-1]
for idx in indices:
    if idx < len(sub):
        row = sub.iloc[idx]
        print(f"  ID {int(row['id']):4d}: {str(row['translation'])[:70]}")

# Save
sub_path = Path(OUTPUT_DIR) / 'submission.csv'
sub.to_csv(sub_path, index=False)
print(f"\nSaved: {sub_path} ({len(sub)} rows)")

Total: 4 translations
Empty: 0 (0.0%)
Lengths: mean=167, median=146, min=138, max=239

Samples:
  ID    0: Thus kārum Kanesh, say to the <big_gap> dātum, our messenger, every si
  ID    2: As soon as you have heard our letter, whether he has sold anything the
  ID    3: I sent our tablet to every single colony and the trading stations. Whe

Saved: /kaggle/working/submission.csv (4 rows)
